# Реальная лабораторная 04 — Student Performance и Wine Quality

Прогнозная постановка, ordinal target и риск семантической утечки.

> **Как работать:** выполняй ячейки последовательно. Сначала сформулируй ожидание, затем запускай код и объясняй результат. Не удаляй строки и не исправляй значения автоматически до выяснения их смысла.

**Данные:** скачиваются из UCI Machine Learning Repository через `download_uci_datasets.py`. Первый запуск требует интернета; после сохранения CSV лабораторная выполняется локально.

## 0. Исследовательский контракт

Запиши единицу наблюдения, целевую аудиторию анализа, момент принятия решения и то, какие выводы допустимы. Разделяй описательное исследование, прогноз и причинные утверждения.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

SEED = 42
RNG = np.random.default_rng(SEED)
DATA_DIR = Path("data/raw")
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 60)

def load_real(slug):
    path = DATA_DIR / f"{slug}.csv"
    if not path.exists():
        raise FileNotFoundError(
            f"Не найден {path}. Сначала выполните: "
            f"python download_uci_datasets.py --only {slug}"
        )
    return pd.read_csv(path, low_memory=False)

def section(title, text):
    display(Markdown(f"### {title}\n\n{text}"))

## Часть A. Student Performance — сценарии момента прогноза

In [ ]:
students = load_real("student_performance")
print(students.shape); display(students.head()); display(students.dtypes.to_frame("dtype"))
colmap = {str(c).lower():c for c in students.columns}
g3 = colmap.get("g3")
assert g3, "Не найден target G3"
for c in [colmap.get("g1"),colmap.get("g2"),g3]:
    if c: students[c] = pd.to_numeric(students[c], errors="coerce")
display(students[[c for c in [colmap.get("g1"),colmap.get("g2"),g3] if c]].describe())

### Сравнение раннего и позднего прогноза

In [ ]:
g1, g2 = colmap.get("g1"), colmap.get("g2")
if g1 and g2:
    display(students[[g1,g2,g3]].corr(method="spearman"))
    fig,ax=plt.subplots(1,2,figsize=(12,4))
    sns.scatterplot(data=students,x=g1,y=g3,alpha=.55,ax=ax[0]); ax[0].set_title("G1 vs G3")
    sns.scatterplot(data=students,x=g2,y=g3,alpha=.55,ax=ax[1]); ax[1].set_title("G2 vs G3")
    plt.tight_layout(); plt.show()
    early_features=[c for c in students.columns if c.lower() not in {"g1","g2","g3"}]
    late_features=[c for c in students.columns if c.lower() != "g3"]
    print("Early forecast excludes:", [g1,g2])
    print("Late forecast may include:", [g1,g2])

## Часть B. Wine Quality — регрессия или ordinal classification

In [ ]:
wine = load_real("wine_quality")
print(wine.shape); display(wine.head()); display(wine.describe(include="all").T)
quality_col = next((c for c in wine.columns if c.lower()=="quality"), None)
assert quality_col, "Не найден quality"
wine[quality_col]=pd.to_numeric(wine[quality_col],errors="coerce")
display(wine[quality_col].value_counts(dropna=False).sort_index().to_frame("count"))

### Хвосты, зависимости и неравные расстояния между оценками

In [ ]:
num_w = wine.select_dtypes(include=np.number)
display(num_w.corr(method="spearman")[[quality_col]].sort_values(quality_col, ascending=False))
fig,ax=plt.subplots(1,2,figsize=(12,4))
q = wine[quality_col].dropna().to_numpy()
if len(q):
    bins = np.arange(np.floor(q.min())-.5, np.ceil(q.max())+1.5, 1)
    ax[0].hist(q, bins=bins, edgecolor="white")
ax[0].set_title("Ordinal target distribution")
if "alcohol" in wine.columns:
    groups=[wine.loc[wine[quality_col]==level,"alcohol"].dropna().to_numpy() for level in sorted(wine[quality_col].dropna().unique())]
    levels=sorted(wine[quality_col].dropna().unique())
    if groups and any(len(g) for g in groups):
        ax[1].boxplot(groups, labels=levels)
    ax[1].set_xlabel("quality"); ax[1].set_ylabel("alcohol"); ax[1].set_title("Alcohol by quality")
plt.tight_layout(); plt.show()

## Самостоятельные задания

1. Для Student Performance сформулируй две разные постановки: прогноз до начала занятий и прогноз после второго периода. Какие переменные допустимы?
2. Оцени, насколько G1/G2 объясняют G3 и почему это не универсальный leakage без определения момента прогнозирования.
3. Для Wine Quality сравни среднее, медиану и распределение по категориям качества.
4. Сравни регрессионную, ordinal- и обычную multiclass постановки. Что теряется в каждой?
5. Исследуй мультиколлинеарность физико-химических признаков и предложи осторожную интерпретацию.

## Итоговый отчёт

В финальной версии notebook должны быть: **вопрос → единица анализа → происхождение и ограничения данных → проверка качества → описательная статистика → закономерности → неопределённость → риски смещения/утечки → выводы → следующие проверки**. Каждый вывод должен ссылаться на таблицу, график или расчёт.